# Healthcare Data Analyst Project
## Python + Pandas + Seaborn + Matplotlib

This notebook analyzes the supplied healthcare datasets:

- Appointment
- Billing
- Doctor
- Patient

### Business objectives
1. Understand patient, doctor and appointment volumes.
2. Identify high-volume doctors and specializations.
3. Analyze billing and revenue patterns.
4. Analyze appointment trends over time.
5. Produce portfolio-ready visualizations and analytical tables.


In [ ]:
# Install dependencies if needed
# Uncomment the next line in a new environment:
# %pip install pandas numpy matplotlib seaborn

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Working directory:", BASE_DIR)
print("Output directory:", OUTPUT_DIR)

## 1. Load the datasets

In [ ]:
files = {
    "appointments": "Appointment(2).csv",
    "billing": "Billing(2).csv",
    "doctors": "Doctor(2).csv",
    "patients": "Patient(2).csv",
}

data = {}
for key, filename in files.items():
    path = os.path.join(BASE_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing {filename}. Keep the CSV files in the same folder as this notebook."
        )
    data[key] = pd.read_csv(path)

appointments = data["appointments"]
billing = data["billing"]
doctors = data["doctors"]
patients = data["patients"]

for name, df in data.items():
    print(f"{name}: {df.shape}")

## 2. Initial data inspection

In [ ]:
for name, df in data.items():
    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)
    display(df.head())
    print("Shape:", df.shape)
    print("Missing values:", int(df.isna().sum().sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))
    display(df.dtypes.to_frame("Data Type"))

## 3. Data cleaning

In [ ]:
appointments = appointments.copy()
billing = billing.copy()
doctors = doctors.copy()
patients = patients.copy()

appointments["Date"] = pd.to_datetime(appointments["Date"], errors="coerce")
appointments["Time"] = pd.to_datetime(appointments["Time"], errors="coerce", utc=True)
billing["Amount"] = pd.to_numeric(billing["Amount"], errors="coerce")

appointments = appointments.drop_duplicates()
billing = billing.drop_duplicates()
doctors = doctors.drop_duplicates()
patients = patients.drop_duplicates()

for df, cols in [
    (doctors, ["DoctorName", "Specialization"]),
    (patients, ["FirstName", "LastName"]),
    (billing, ["Items"]),
]:
    for col in cols:
        df[col] = df[col].astype("string").str.strip()

data = {
    "appointments": appointments,
    "billing": billing,
    "doctors": doctors,
    "patients": patients,
}

print("Cleaning completed.")

## 4. Data quality check

In [ ]:
quality = []

for name, df in data.items():
    quality.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Missing_Values": int(df.isna().sum().sum()),
    })

quality_df = pd.DataFrame(quality)
display(quality_df)

## 5. Create analytical tables with Pandas

In [ ]:
appointment_detail = (
    appointments
    .merge(patients, on="PatientID", how="left")
    .merge(doctors, on="DoctorID", how="left")
)

patient_billing = (
    billing.groupby("PatientID", as_index=False)
    .agg(
        Total_Billing=("Amount", "sum"),
        Number_of_Invoices=("InvoiceID", "nunique")
    )
)

doctor_appointments = (
    appointment_detail
    .groupby(["DoctorID", "DoctorName", "Specialization"], dropna=False, as_index=False)
    .agg(Appointment_Count=("AppointmentID", "nunique"))
    .sort_values("Appointment_Count", ascending=False)
)

specialization_summary = (
    appointment_detail
    .groupby("Specialization", dropna=False, as_index=False)
    .agg(
        Appointment_Count=("AppointmentID", "nunique"),
        Unique_Doctors=("DoctorID", "nunique"),
        Unique_Patients=("PatientID", "nunique")
    )
    .sort_values("Appointment_Count", ascending=False)
)

billing_summary = (
    billing
    .groupby("Items", dropna=False, as_index=False)
    .agg(
        Total_Revenue=("Amount", "sum"),
        Average_Bill=("Amount", "mean"),
        Invoice_Count=("InvoiceID", "nunique")
    )
    .sort_values("Total_Revenue", ascending=False)
)

monthly_appointments = (
    appointments.dropna(subset=["Date"])
    .assign(Month=appointments.dropna(subset=["Date"])["Date"].dt.to_period("M").astype(str))
    .groupby("Month", as_index=False)
    .agg(Appointment_Count=("AppointmentID", "nunique"))
    .sort_values("Month")
)

display(doctor_appointments.head(10))
display(specialization_summary.head(10))
display(billing_summary.head(10))

## 6. Healthcare KPIs

In [ ]:
kpis = {
    "Total Patients": patients["PatientID"].nunique(),
    "Total Doctors": doctors["DoctorID"].nunique(),
    "Total Appointments": appointments["AppointmentID"].nunique(),
    "Total Invoices": billing["InvoiceID"].nunique(),
    "Total Revenue": billing["Amount"].sum(),
    "Average Bill Amount": billing["Amount"].mean(),
    "Maximum Bill Amount": billing["Amount"].max(),
    "Minimum Bill Amount": billing["Amount"].min(),
    "Average Appointments per Doctor": (
        appointments["AppointmentID"].nunique() / appointments["DoctorID"].nunique()
        if appointments["DoctorID"].nunique() else np.nan
    ),
    "Average Appointments per Patient": (
        appointments["AppointmentID"].nunique() / appointments["PatientID"].nunique()
        if appointments["PatientID"].nunique() else np.nan
    ),
}

kpi_df = pd.DataFrame({"KPI": kpis.keys(), "Value": kpis.values()})
display(kpi_df)

## 7. Visualization — Monthly appointment trend

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_appointments, x="Month", y="Appointment_Count", marker="o")
plt.title("Monthly Appointment Trend")
plt.xlabel("Month")
plt.ylabel("Number of Appointments")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_monthly_appointments.png"), dpi=150)
plt.show()

## 8. Visualization — Top doctors

In [ ]:
top_doctors = doctor_appointments.head(10).sort_values("Appointment_Count")

plt.figure(figsize=(10, 6))
sns.barplot(data=top_doctors, x="Appointment_Count", y="DoctorName")
plt.title("Top 10 Doctors by Appointment Volume")
plt.xlabel("Appointments")
plt.ylabel("Doctor")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_top_doctors.png"), dpi=150)
plt.show()

## 9. Visualization — Revenue by billing item

In [ ]:
top_revenue = billing_summary.head(10).sort_values("Total_Revenue")

plt.figure(figsize=(10, 7))
sns.barplot(data=top_revenue, x="Total_Revenue", y="Items")
plt.title("Top Billing Items by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Billing Item")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_revenue_by_item.png"), dpi=150)
plt.show()

## 10. Visualization — Specialization demand

In [ ]:
top_specializations = specialization_summary.head(10).sort_values("Appointment_Count")

plt.figure(figsize=(10, 7))
sns.barplot(data=top_specializations, x="Appointment_Count", y="Specialization")
plt.title("Top Specializations by Appointment Volume")
plt.xlabel("Appointments")
plt.ylabel("Specialization")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_specialization_volume.png"), dpi=150)
plt.show()

## 11. Visualization — Billing distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=billing, x="Amount", bins=30, kde=True)
plt.title("Distribution of Billing Amounts")
plt.xlabel("Bill Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_billing_distribution.png"), dpi=150)
plt.show()

## 12. Visualization — Appointments by year

In [ ]:
yearly = (
    appointments.dropna(subset=["Date"])
    .assign(Year=appointments.dropna(subset=["Date"])["Date"].dt.year)
    .groupby("Year", as_index=False)
    .agg(Appointments=("AppointmentID", "nunique"))
)

plt.figure(figsize=(9, 5))
sns.barplot(data=yearly, x="Year", y="Appointments")
plt.title("Appointments by Year")
plt.xlabel("Year")
plt.ylabel("Appointments")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_appointments_by_year.png"), dpi=150)
plt.show()

## 13. Business insights

In [ ]:
top_doctor = doctor_appointments.iloc[0] if not doctor_appointments.empty else None
top_spec = specialization_summary.iloc[0] if not specialization_summary.empty else None
top_item = billing_summary.iloc[0] if not billing_summary.empty else None

print("KEY BUSINESS INSIGHTS")
print("-" * 50)
print(f"Total revenue: {billing['Amount'].sum():,.2f}")
print(f"Average bill: {billing['Amount'].mean():,.2f}")

if top_doctor is not None:
    print(f"Highest appointment-volume doctor: {top_doctor['DoctorName']} "
          f"({int(top_doctor['Appointment_Count']):,} appointments)")

if top_spec is not None:
    print(f"Highest-demand specialization: {top_spec['Specialization']} "
          f"({int(top_spec['Appointment_Count']):,} appointments)")

if top_item is not None:
    print(f"Highest-revenue billing item: {top_item['Items']} "
          f"({top_item['Total_Revenue']:,.2f})")

## 14. Export results

In [ ]:
tables = {
    "appointment_detail": appointment_detail,
    "patient_billing": patient_billing,
    "doctor_appointments": doctor_appointments,
    "specialization_summary": specialization_summary,
    "billing_summary": billing_summary,
    "monthly_appointments": monthly_appointments,
    "kpis": kpi_df,
}

for name, df in tables.items():
    df.to_csv(os.path.join(OUTPUT_DIR, f"{name}.csv"), index=False)

for name, df in data.items():
    df.to_csv(os.path.join(OUTPUT_DIR, f"cleaned_{name}.csv"), index=False)

print(f"All analytical tables exported to: {OUTPUT_DIR}")

# Conclusion

This project demonstrates a complete **Data Analyst workflow in Python**:

- Data loading with Pandas
- Data cleaning and validation
- Relational joins with `merge()`
- KPI creation
- `groupby()` and aggregations
- Date/time analysis
- Revenue analysis
- Exploratory Data Analysis
- Seaborn and Matplotlib visualization
- Business insight generation
- Exporting analysis-ready datasets

### Resume-ready project
**Healthcare Data Analytics Project:** Analyzed patient, doctor, appointment and billing data using Python, Pandas, Seaborn and Matplotlib; performed data cleaning, relational joins, KPI analysis and exploratory visualization to identify appointment trends, high-volume doctors, specialization demand and revenue-driving services.
